# PokeZero playground

A hands-on tour of the repo: step through a game, pit agents against each other, look inside the
encoder and network, edit the config and run a short training loop, then watch a replay.
Works locally (Jupyter / VS Code) and on Google Colab.

**Why is the game CPU-only, and is a GPU best for training?**

* The *game engine* (`kaggle_environments.envs.cabt`) is a native library that only runs on the
  CPU. Every self-play game, and every search simulation, is CPU work. That is the bottleneck.
* The *network* can use a GPU. `TrainConfig.device` (`cpu` / `cuda` / `mps` / `auto`) moves the
  gradient steps there; self-play and the submission agent stay on CPU.
* With a network this small, a GPU is mostly idle and waiting on the engine, so more CPU cores buy
  more games per dollar. A GPU starts to pay off with a much bigger network, batched inference
  across many parallel games, or behaviour cloning from big replay exports.

The notebook is saved without outputs. It prints option *types* and shapes, never card names or text.

## 0. Setup

In [ ]:
import sys

if "google.colab" in sys.modules:
    # Colab: install the package from GitHub (a private repo needs a token in the URL).
    %pip install -q "pokezero @ git+https://github.com/dhruvcharan13/PokeZero.git"

In [ ]:
import json
import random
import tempfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from pokezero.agents import HeuristicAgent, RandomAgent
from pokezero.agents.az import PolicyAgent
from pokezero.config import load_config, to_dict
from pokezero.env.cabt import CabtEnv
from pokezero.eval.arena import format_report, play_match, round_robin
from pokezero.model.encoding import encode
from pokezero.model.net import NetConfig, PokeNet
from pokezero.train.learner import resolve_device

DEVICE = resolve_device("auto")  # cuda > mps > cpu
print("training device:", DEVICE, "| game engine: always CPU")

## 1. Step through a game

`CabtEnv.reset()` returns a `Decision`: who is acting, what kind of prompt it is, how many options
they must pick (`min_count`..`max_count`), and the observation that player sees (their own hand
plus the public board; the opponent's hand is hidden).

In [ ]:
env = CabtEnv()
d = env.reset(seed=0)
print("acting player:", d.player, "| select_type:", d.select_type, "| context:", d.context)
print("pick", d.min_count, "to", d.max_count, "of", d.num_options, "options")
print("observation keys:", sorted(d.obs))
print("current keys:", sorted(d.obs["current"]))
print("option types:", Counter(op["type"] for op in d.options))
print("first option:", d.options[0])

In [ ]:
# Play the first 15 decisions by hand (random legal picks) and watch the prompt change.
rng = random.Random(1)
for step in range(15):
    k = rng.randint(d.min_count, min(d.max_count, d.num_options))
    picks = rng.sample(range(d.num_options), k)
    kinds = sorted({op["type"] for op in d.options})
    print(
        f"{step:2d} player={d.player} select={d.select_type} ctx={d.context:2d} "
        f"n={d.num_options:2d} pick {picks} option types={kinds}"
    )
    nxt = env.step(picks)
    if nxt is None:
        break
    d = nxt
env.close()

## 2. Agents against each other

`RandomAgent`, `HeuristicAgent` (hand-written rules) and `PolicyAgent` (a network, here untrained).
`play_match` alternates seats. Try changing the game counts, or write your own agent: anything with
`reset()` and `act(decision) -> list[int]` works.

In [ ]:
net = PokeNet().eval()
print(f"network parameters: {net.num_parameters():,}")

m = play_match(HeuristicAgent(0), RandomAgent(1), games=10)
print(
    f"heuristic vs random: {m.wins}W {m.losses}L {m.draws}D "
    f"({m.games / m.seconds:.1f} games/s on this machine)"
)

rr = round_robin([RandomAgent(0), HeuristicAgent(0), PolicyAgent(net, "untrained-net")], 6)
print(format_report(rr))

## 3. Inside the encoder and the network

`encode` turns one prompt into arrays: global scalars, board slots (active + bench per side), hand,
discards, and one feature row per option plus a final STOP row. The net scores each option row.

In [ ]:
env = CabtEnv()
d = env.reset(seed=0)
while d is not None and d.num_options < 3:  # find a prompt with a real choice
    d = env.step(list(range(d.min_count)))
enc = encode(d)
for name in ["glob", "slot_ids", "slot_feat", "hand_ids", "opt_feat", "opt_card", "mask"]:
    print(f"{name:10s} {getattr(enc, name).shape}")
print("legal rows:", np.flatnonzero(enc.mask).tolist(), "(last row = STOP)")

priors, value = net.predict([enc])
print("priors:", np.round(priors[0], 3))
print("value (win prob estimate in [-1, 1]):", float(value[0]))
env.close()

In [ ]:
# Where do the parameters live?
for name, p in net.named_parameters():
    if p.numel() > 100_000:
        print(f"{name:40s} {tuple(p.shape)}  {p.numel():,}")

## 4. Edit the config and train

Configs are plain dataclasses (see `configs/default.yaml` and `configs/smoke.yaml`). Anything can be
overridden by dotted key, exactly like `pokezero train a.b=value` on the command line.
The smoke config finishes one tiny iteration in about a minute on CPU. Raise `iterations`,
`selfplay.games_per_iter` and `train.steps_per_iter` for something more meaningful.

Note: self-play uses PUCT search over the engine's forward model, which leaks the opponent's hand
(research-only, see `docs/sdk-findings.md`). The submission agent is the search-free policy net.

In [ ]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs").exists())
cfg = load_config(
    REPO / "configs" / "smoke.yaml",
    [
        "run_name=notebook",
        f"artifact_root={tempfile.mkdtemp(prefix='pokezero-runs-')}",
        "iterations=3",
        "selfplay.games_per_iter=2",
        "selfplay.search.simulations=8",
        "train.steps_per_iter=40",
        "eval.games=4",
        "eval.baseline_games=4",
        f"train.device={DEVICE.type}",
    ],
)
print(json.dumps(to_dict(cfg)["train"], indent=2))

In [ ]:
from pokezero.storage import LocalStore
from pokezero.train.loop import run_training

store = LocalStore(Path(cfg.artifact_root) / cfg.run_name)
last = run_training(cfg, store)  # prints one JSON line per iteration

In [ ]:
rows = [json.loads(line) for line in store.get_bytes("metrics.jsonl").decode().splitlines()]
its = [r["step"] for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].plot(its, [r["policy_loss"] for r in rows], marker="o", label="policy")
ax[0].plot(its, [r["value_loss"] for r in rows], marker="o", label="value")
ax[0].set(title="losses", xlabel="iteration")
ax[0].legend()
for k in ("candidate_vs_random", "candidate_vs_heuristic", "h2h_win_rate"):
    ax[1].plot(its, [r[k] for r in rows], marker="o", label=k)
ax[1].set(title="win rates", xlabel="iteration", ylim=(0, 1))
ax[1].legend(fontsize=7)
ax[2].plot(its, [r["iter_seconds"] for r in rows], marker="o", label="iteration")
ax[2].plot(its, [r["selfplay_seconds"] for r in rows], marker="o", label="self-play")
ax[2].set(title="seconds (self-play dominates)", xlabel="iteration")
ax[2].legend()
plt.tight_layout()

In [ ]:
# Load the champion checkpoint and play it against the baselines.
ck = torch.load(store.local_path("champion.pt"), map_location="cpu", weights_only=True)
champ = PokeNet(NetConfig(**ck["net_cfg"]))
champ.load_state_dict(ck["state_dict"])
champ.eval()
m = play_match(PolicyAgent(champ, "champion"), RandomAgent(0), games=10)
print(f"champion (iteration {ck['iteration']}) vs random: {m.win_rate:.0%}")

## 5. Render a replay

Kaggle's own replay viewer, embedded inline. The viewer is a spectator view and shows both hands.
Replays are public on the competition, but don't commit or share these files from the notebook.

In [ ]:
import html as _html

from IPython.display import HTML, display

from pokezero.viz import render_game_html

page, rewards = render_game_html(HeuristicAgent(0), PolicyAgent(champ, "champion"))
print("seat 0 (heuristic) reward:", rewards[0])
display(HTML(f'<iframe srcdoc="{_html.escape(page)}" width="100%" height="700"></iframe>'))

## 6. Things to try

* Swap `HeuristicAgent` for your own rules in `pokezero/agents/heuristic.py` and re-run section 2.
* Change `net.*` sizes in the config and compare parameter counts and speed (section 3).
* Compare `train.device=cpu` with `auto` on a GPU runtime and a larger `train.batch_size` /
  `net.hidden`; check `selfplay_seconds` against `iter_seconds` to see where time goes.
* Turn on Weights & Biases with `wandb.enabled=true` (needs `WANDB_API_KEY`).
* Package a checkpoint for submission with `pokezero package --checkpoint <path>`.